# Multi-NIfTI Viewer for MRI2CT Comparisons
This notebook allows you to interactively compare the Input MRI, Ground Truth CT, UNet prediction, and Anatomix prediction.

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, RadioButtons, HBox, VBox, Layout
from IPython.display import display

# Set subject directory here
VIZ_DIR = "../viz"
SUBJECT_ID = "1ABA005"  # Change this to the subject you want to view
SUBJ_PATH = os.path.join(VIZ_DIR, SUBJECT_ID)

print(f"Looking in: {SUBJ_PATH}")

In [ ]:
def load_nifti(filename):
    path = os.path.join(SUBJ_PATH, filename)
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return None
    img = nib.load(path)
    return img.get_fdata()


# Load the 4 volumes
vol_mri = load_nifti("input_mri.nii.gz")
vol_gt = load_nifti("gt_ct.nii.gz")
vol_unet = load_nifti("pred_ct_unet.nii.gz")
vol_amix = load_nifti("pred_ct_amix.nii.gz")

if vol_mri is not None:
    print(f"Volumes loaded successfully. Shape: {vol_mri.shape}")

In [ ]:
def apply_window(vol, window_level, window_width):
    if vol is None:
        return None
    vmin = window_level - (window_width / 2)
    vmax = window_level + (window_width / 2)
    return np.clip(vol, vmin, vmax)


def show_slices(z, plane, ct_window):
    if vol_mri is None:
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    axes = axes.flatten()

    # Configure Windowing based on preset
    if ct_window == "Soft Tissue":
        wl, ww = 40, 400
    elif ct_window == "Bone":
        wl, ww = 400, 1500
    elif ct_window == "Lung":
        wl, ww = -600, 1500
    else:  # Default/Full Range
        wl, ww = 0, 2048

    # Extract slices based on chosen plane
    def get_slice(vol):
        if vol is None:
            return np.zeros((10, 10))
        if plane == "Axial":
            return np.rot90(vol[:, :, z])
        elif plane == "Sagittal":
            return np.rot90(vol[z, :, :])
        elif plane == "Coronal":
            return np.rot90(vol[:, z, :])

    slice_mri = get_slice(vol_mri)
    slice_gt = apply_window(get_slice(vol_gt), wl, ww)
    slice_unet = apply_window(get_slice(vol_unet), wl, ww)
    slice_amix = apply_window(get_slice(vol_amix), wl, ww)

    # Plot MRI (Self-normalized for visualization)
    axes[0].imshow(slice_mri, cmap="gray")
    axes[0].set_title("Input MRI", fontsize=14)

    # Plot Ground Truth CT
    axes[1].imshow(slice_gt, cmap="gray", vmin=wl - (ww / 2), vmax=wl + (ww / 2))
    axes[1].set_title("Ground Truth CT", fontsize=14)

    # Plot UNet Baseline
    if vol_unet is not None:
        axes[2].imshow(slice_unet, cmap="gray", vmin=wl - (ww / 2), vmax=wl + (ww / 2))
        axes[2].set_title("UNet Baseline Prediction", fontsize=14)
    else:
        axes[2].set_title("UNet Not Found", fontsize=14)

    # Plot Anatomix
    if vol_amix is not None:
        axes[3].imshow(slice_amix, cmap="gray", vmin=wl - (ww / 2), vmax=wl + (ww / 2))
        axes[3].set_title("Anatomix Prediction", fontsize=14)
    else:
        axes[3].set_title("Anatomix Not Found", fontsize=14)

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


if vol_mri is not None:
    # Determine max slices per plane
    shape = vol_mri.shape
    max_z = {"Axial": shape[2] - 1, "Sagittal": shape[0] - 1, "Coronal": shape[1] - 1}

    # Widgets
    plane_selector = RadioButtons(options=["Axial", "Sagittal", "Coronal"], value="Axial", description="Plane:")
    window_selector = RadioButtons(options=["Default", "Soft Tissue", "Bone", "Lung"], value="Default", description="CT Window:")
    z_slider = IntSlider(min=0, max=max_z["Axial"], value=max_z["Axial"] // 2, description="Slice:", layout=Layout(width="800px"))

    def update_z_max(*args):
        z_slider.max = max_z[plane_selector.value]
        z_slider.value = z_slider.max // 2

    plane_selector.observe(update_z_max, "value")

    ui = VBox([HBox([plane_selector, window_selector]), z_slider])
    out = interact(show_slices, z=z_slider, plane=plane_selector, ct_window=window_selector)
    display(ui)
